# Unidad 40. Distribuciones y calidad de una variable

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 5–7 horas

> Una distribución resume cómo se reparten los valores de una variable. Antes de interpretarla, hay que saber qué mide la variable, cuántas observaciones hay y qué problemas de calidad contienen los datos.

## Resultados de aprendizaje

Al terminar podrás:

- revisar valores ausentes antes de representar una variable numérica;
- crear e interpretar histogramas, estimaciones KDE y funciones de distribución acumulada (ECDF);
- justificar una elección de intervalos y distinguir frecuencia, proporción y densidad;
- comparar distribuciones sin convertir una pauta visual en una afirmación causal.

**Prerequisitos:** unidad 39 y el manejo básico de `DataFrame` de Pandas.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")


def ruta_datos(nombre):
    candidatas = [
        Path("datos") / "seaborn" / nombre,
        Path("..") / "datos" / "seaborn" / nombre,
    ]
    for candidata in candidatas:
        if candidata.exists():
            return candidata
    raise FileNotFoundError(
        f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso."
    )

mediciones = pd.read_csv(ruta_datos("40_mediciones_calidad_aire.csv"))
mediciones.head()

,lectura_id,zona,sensor,pm25_ug_m3,temperatura_c
0,1,Norte,N-01,7.8,16.2
1,2,Norte,N-01,8.4,16.7
2,3,Norte,N-01,9.1,17.1
3,4,Norte,N-01,10.3,17.4
4,5,Norte,N-01,11.2,18.0


## 1. Antes del gráfico: qué observamos y qué falta

El archivo contiene lecturas **simuladas** de PM2.5 de tres zonas. Una observación no es una medición real ni permite hacer afirmaciones sobre una ciudad. La variable `pm25_ug_m3` es numérica y continua: puede tomar valores decimales.

Los valores ausentes no son ceros. Un cero indicaría una concentración medida igual a cero; un valor ausente significa que desconocemos la lectura. Para explorar esta variable eliminaremos únicamente las filas sin PM2.5, conservando el resto de columnas y documentando la decisión.

In [ ]:
mediciones.info()

ausentes = mediciones.isna().sum()
print(ausentes)

pm25_limpias = mediciones.dropna(subset=["pm25_ug_m3"]).copy()
print(f"Filas originales: {len(mediciones)}")
print(f"Filas para PM2.5: {len(pm25_limpias)}")
pm25_limpias.describe()

La eliminación anterior es razonable para un ejemplo breve porque solo faltan dos lecturas y estamos estudiando una única variable. En otro contexto podrías investigar por qué faltan datos, informar del porcentaje perdido o usar un método de imputación justificado. Nunca ocultes esa decisión.

## 2. Histograma: valores agrupados en intervalos

Un histograma divide el eje horizontal en **intervalos** o *bins* y cuenta cuántas observaciones caen en cada uno. Con pocos intervalos se pierde detalle; con demasiados, el ruido de una muestra pequeña puede parecer una pauta importante. No existe un número de intervalos universalmente correcto: pruébalo, explica tu elección y mira los datos originales.

Con `stat="count"`, la altura es el número de observaciones. No confundas ese conteo con una probabilidad ni con una densidad.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    bins=8,
    stat="count",
    color="#4477AA",
    edgecolor="white",
    ax=ax,
)
ax.set(
    title="Distribución simulada de PM2.5",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Número de lecturas",
)
plt.show()

### Comprueba que lo has entendido

Antes de modificar el código, predice qué ocurrirá si cambias `bins=8` por `bins=16`: ¿cambiará el número total de lecturas? ¿Cambiará el aspecto del gráfico? Ejecuta la variante y responde.

El total no cambia, porque las observaciones son las mismas. Sí cambia el aspecto: al haber intervalos más estrechos, algunos pueden contener muy pocas observaciones. Por eso una conclusión no debe depender de una única elección de bins.

## 3. KDE: una curva suavizada, no datos adicionales

Una estimación de densidad por núcleo o **KDE** dibuja una curva suave que resume dónde se concentran los valores. Es útil para ver una forma general, pero depende de un parámetro de suavizado y puede sugerir detalles que los datos no sostienen. No la interpretes como una frecuencia exacta ni como la distribución verdadera de una población.

En el gráfico siguiente se conserva el histograma con `stat="density"`. Ahora el área total de las barras y de la curva se interpreta como 1; la altura ya no representa un número de lecturas.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    bins=8,
    stat="density",
    color="#AABBDD",
    edgecolor="white",
    ax=ax,
)
sns.kdeplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    color="#225588",
    linewidth=2,
    ax=ax,
)
ax.set(
    title="Histograma y KDE de PM2.5",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Densidad",
)
plt.show()

## 4. ECDF: proporción acumulada sin elegir intervalos

La **función de distribución acumulada empírica** (ECDF) muestra, para cada valor horizontal, qué proporción de observaciones es menor o igual que él. No exige elegir bins ni suavizado. La curva sube en escalones y termina en 1.

Por ejemplo, si la ECDF vale aproximadamente 0,50 en 18 µg/m³, alrededor de la mitad de las lecturas limpias no supera ese valor. No afirma nada sobre lecturas futuras ni sobre zonas no observadas.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.ecdfplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    hue="zona",
    linewidth=2,
    ax=ax,
)
ax.set(
    title="Proporción acumulada de lecturas por zona",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Proporción acumulada",
)
ax.legend(title="Zona")
plt.show()

Una curva más desplazada a la derecha indica valores generalmente mayores dentro de esa muestra. Para interpretar la diferencia necesitarías saber si los sensores son comparables, cuándo se tomaron las lecturas, cuántas faltan y qué otras variables intervienen.

## 5. Funciones de nivel de figura: `displot`

`sns.displot` crea una figura completa para distribuciones. Puede construir histogramas, KDE o ECDF y, más adelante, dividirlos en paneles. Como en `relplot`, el objeto resultante administra su propia figura. Úsalo cuando no necesitas insertarlo en unos ejes de Matplotlib ya existentes.

In [ ]:
grafico_distribucion = sns.displot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    col="zona",
    kind="hist",
    bins=5,
    color="#4477AA",
    height=3.2,
    aspect=1,
)
grafico_distribucion.set_axis_labels("PM2.5 (µg/m³)", "Número de lecturas")
grafico_distribucion.figure.suptitle("Histogramas por zona simulada", y=1.05)
plt.show()

## Práctica guiada · comparar una decisión de intervalo

Construye dos histogramas de `pm25_ug_m3` en una misma figura, con `bins=5` y `bins=12`. Usa `sharey=True`, pon títulos claros y conserva el eje vertical como número de lecturas. Después escribe dos frases: una sobre una pauta que aparezca en ambos paneles y otra sobre un detalle que cambie al modificar los bins.

La comparación te ayuda a comprobar si una conclusión depende excesivamente de una elección de diseño.

In [ ]:
# Completa la práctica guiada aquí.
# Crea fig, ejes = plt.subplots(1, 2, sharey=True, figsize=(10, 4)).
fig, ejes = plt.subplots(1, 2, sharey=True, figsize=(10, 4), layout="constrained")

for ax, n_bins in zip(ejes, [5, 12]):
    sns.histplot(
        data=pm25_limpias,
        x="pm25_ug_m3",
        bins=n_bins,
        stat="count",
        color="#4477AA",
        edgecolor="white",
        ax=ax,
    )
    ax.set(
        title=f"Distribución de PM2.5 (bins={n_bins})",
        xlabel="PM2.5 (µg/m³)",
        ylabel="Número de lecturas",
    )

fig.suptitle("Efecto del número de intervalos en el histograma de PM2.5")
plt.show()

# Pauta común: En ambos paneles se observa que la mayor parte de las lecturas se concentran en
# valores bajos e intermedios (por debajo de 20 µg/m³), con una cola hacia valores más altos.
# Detalle que cambia: Con bins=12 aparecen huecos o escalones irregulares en las barras debido al
# tamaño reducido de la muestra, mientras que con bins=5 la forma general se ve mucho más suave.

- **[U40.2]** `fig, ejes = plt.subplots(1, 2, sharey=True, figsize=(10, 4), layout="constrained")` crea una figura con dos paneles horizontales que comparten la misma escala vertical (`sharey=True`) para que las alturas sean comparables de forma honesta,
- **[U40.2]** `for ax, n_bins in zip(ejes, [5, 12]):` recorre simultáneamente cada eje (`ejes[0]` y `ejes[1]`) junto con su número de intervalos correspondiente (`5` y `12`),
- **[U40.2]** `sns.histplot(data=pm25_limpias, x="pm25_ug_m3", bins=n_bins, stat="count", color="#4477AA", edgecolor="white", ax=ax)` dibuja en nivel de ejes el histograma de conteos (`stat="count"`) con bordes blancos para separar visualmente las barras,
- **[U40.2]** `ax.set(...)` y `fig.suptitle(...)` asignan el título de cada panel indicando los `bins`, las etiquetas con unidades (`µg/m³`) y el encabezado global antes de mostrar con `plt.show()`.

## Ejercicios

Usa siempre `pm25_limpias` para gráficos de PM2.5, salvo que el enunciado pida explícitamente analizar los valores ausentes.

### Ejercicio 40.01 · Informe mínimo de calidad

Crea una tabla con el número de filas, valores ausentes y porcentaje de valores ausentes de cada columna. Verifica con una aserción que `pm25_ug_m3` tiene dos valores ausentes. Explica por qué no sería correcto reemplazarlos automáticamente por cero.

In [ ]:
# Escribe aquí tu solución del ejercicio 40.01.
informe_calidad = pd.DataFrame({
    "filas_totales": len(mediciones),
    "valores_ausentes": mediciones.isna().sum(),
    "porcentaje_ausentes": (mediciones.isna().mean() * 100).round(2),
})
display(informe_calidad)

assert mediciones["pm25_ug_m3"].isna().sum() == 2, (
    f"Se esperaban 2 ausentes en pm25_ug_m3, pero hay {mediciones['pm25_ug_m3'].isna().sum()}"
)

# Explicación: Reemplazar los valores ausentes (NaN) por cero sería incorrecto porque un cero
# representa una medición real de 0 µg/m³ de contaminación (aire completamente limpio), mientras
# que NaN indica que la lectura se perdió o no se registró; imputar ceros distorsionaría hacia
# abajo la media, los cuantiles y la forma de la distribución.

- **[U40.1]** `informe_calidad = pd.DataFrame({...})` construye una tabla resumen indexada por columna con el conteo total de filas (`len(mediciones)`), el número de nulos (`.isna().sum()`) y el porcentaje de nulos (`.isna().mean() * 100`),
- **[U40.1]** `display(informe_calidad)` muestra la tabla de calidad de datos formateada en pantalla,
- **[U40.1]** `assert mediciones["pm25_ug_m3"].isna().sum() == 2, ...` verifica explícitamente mediante aserción que existen exactamente 2 lecturas ausentes en `pm25_ug_m3`, recordando que `NaN` nunca equivale a un valor medido igual a cero.

### Ejercicio 40.02 · Histograma que se pueda leer

Crea un histograma de `temperatura_c` con seis intervalos. Elige un color, añade bordes claros, título y etiquetas con unidades. Indica en un comentario qué representa la altura de cada barra.

In [ ]:
# Escribe aquí tu solución del ejercicio 40.02.
temp_limpia = mediciones.dropna(subset=["temperatura_c"]).copy()

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    data=temp_limpia,
    x="temperatura_c",
    bins=6,
    stat="count",
    color="#2E8B57",
    edgecolor="white",
    ax=ax,
)
ax.set(
    title="Distribución simulada de temperatura (6 intervalos)",
    xlabel="Temperatura (°C)",
    ylabel="Número de lecturas",
)
plt.show()

# La altura de cada barra representa el conteo absoluto (frecuencia) de lecturas registradas
# dentro de cada uno de los seis intervalos de temperatura.

- **[U40.1]** `temp_limpia = mediciones.dropna(subset=["temperatura_c"]).copy()` filtra posibles ausentes únicamente en `temperatura_c` creando una copia independiente para no perder filas válidas de temperatura que tuvieran `NaN` en PM2.5,
- **[U40.2]** `sns.histplot(data=temp_limpia, x="temperatura_c", bins=6, stat="count", color="#2E8B57", edgecolor="white", ax=ax)` dibuja en nivel de ejes el histograma dividido en exactamente 6 intervalos (`bins=6`) con bordes blancos nítidos,
- **[U40.2]** `ax.set(...)` y `plt.show()` establecen el título, las unidades (`°C`) y la etiqueta vertical de conteo absoluto (`Número de lecturas`).

### Ejercicio 40.03 · Frecuencia frente a densidad

Representa PM2.5 con histograma y KDE en los mismos ejes. Configura el histograma con `stat="density"` y explica por qué el eje vertical ya no debe llamarse «Número de lecturas».

In [ ]:
# Escribe aquí tu solución del ejercicio 40.03.
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    bins=8,
    stat="density",
    color="#AABBDD",
    edgecolor="white",
    ax=ax,
)
sns.kdeplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    color="#225588",
    linewidth=2,
    ax=ax,
)
ax.set(
    title="Histograma de densidad y curva KDE de PM2.5",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Densidad",
)
plt.show()

# El eje vertical no debe llamarse «Número de lecturas» porque con stat="density" las barras se
# normalizan para que el área total sume 1 (igual que la curva KDE); la altura mide densidad de
# probabilidad por unidad del eje X (conteo dividido entre total de datos y ancho del bin), no conteos.

- **[U40.3]** `sns.histplot(..., bins=8, stat="density", color="#AABBDD", edgecolor="white", ax=ax)` dibuja el histograma normalizado por área (`stat="density"`) para que su escala vertical coincida con la de la estimación KDE,
- **[U40.3]** `sns.kdeplot(data=pm25_limpias, x="pm25_ug_m3", color="#225588", linewidth=2, ax=ax)` superpone sobre el mismo eje `ax` la curva suave de densidad por núcleo (`KDE`),
- **[U40.3]** `ax.set(..., ylabel="Densidad")` y `plt.show()` etiquetan correctamente el eje vertical como `Densidad` (donde el área total suma `1`, no el conteo de observaciones).

### Ejercicio 40.04 · Pregunta acumulada

Con una ECDF por zona, responde: **«¿Qué zonas tienen todas sus lecturas limpias de PM2.5 por debajo de 20 µg/m³?»** Comprueba primero tu respuesta con una operación de Pandas. No extrapoles la respuesta fuera de este conjunto simulado.

In [ ]:
# Escribe aquí tu solución del ejercicio 40.04.
# 1. Comprobación exacta en Pandas antes de graficar
max_por_zona = pm25_limpias.groupby("zona")["pm25_ug_m3"].max()
zonas_bajo_20 = max_por_zona[max_por_zona < 20].index.tolist()
print("Máximo por zona:\n", max_por_zona)
print("\nZonas con todas sus lecturas limpias < 20 µg/m³:", zonas_bajo_20)

# 2. Visualización con ECDF por zona y línea de referencia en 20 µg/m³
fig, ax = plt.subplots(figsize=(7.5, 4.2))
sns.ecdfplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    hue="zona",
    linewidth=2,
    ax=ax,
)
ax.axvline(20, color="#333333", linestyle="--", linewidth=1.2, label="Umbral 20 µg/m³")
ax.set(
    title="Distribución acumulada (ECDF) de PM2.5 por zona",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Proporción acumulada",
)
ax.legend(title="Zona / Referencia")
plt.show()

# Respuesta: En este conjunto simulado, únicamente la zona 'Norte' alcanza la proporción acumulada
# de 1.0 (100 % de sus lecturas limpias) completamente por debajo de 20 µg/m³.

- **[U40.4]** `max_por_zona = pm25_limpias.groupby("zona")["pm25_ug_m3"].max()` calcula el valor máximo registrado en cada zona para comprobar numéricamente qué zonas tienen el 100 % de sus datos por debajo de `20 µg/m³`,
- **[U40.4]** `sns.ecdfplot(data=pm25_limpias, x="pm25_ug_m3", hue="zona", linewidth=2, ax=ax)` traza en nivel de ejes las funciones de distribución acumulada empírica (`0` a `1`) de cada zona sin depender de intervalos arbitrarios,
- **[U40.4]** `ax.axvline(20, color="#333333", linestyle="--", ...)` añade una línea vertical discontinua en `20 µg/m³` para verificar visualmente qué curva llega a `1.0` a la izquierda del umbral (`Norte`),
- **[U40.4]** `ax.set(...)`, `ax.legend(...)` y `plt.show()` completan las etiquetas, unidades y leyenda del gráfico.

### Ejercicio 40.05 · Miniproyecto: distribución por zona

Elabora una figura que permita comparar la distribución de PM2.5 entre las tres zonas simuladas. Puedes usar una ECDF con `hue`, un `displot` con paneles o una combinación razonada de histograma y KDE. Debe incluir unidades, una leyenda o títulos de panel y una breve justificación de tu elección.

Escribe tres conclusiones: una pauta visible, una limitación de los datos y una comprobación adicional que necesitarías antes de tomar una decisión real.

In [ ]:
# Escribe aquí tu solución del ejercicio 40.05.
# Elección justificada: Se utiliza ecdfplot con hue="zona" porque permite comparar las tres
# distribuciones superpuestas en los mismos ejes sin depender de la elección de bins ni ocultar
# el tamaño reducido de cada subgrupo.
fig, ax = plt.subplots(figsize=(7.5, 4.2), layout="constrained")
sns.ecdfplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    hue="zona",
    linewidth=2.2,
    ax=ax,
)
ax.set(
    title="Comparación de la distribución acumulada de PM2.5 por zona",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Proporción acumulada",
)
ax.legend(title="Zona")
plt.show()

- **[U40.4]** `fig, ax = plt.subplots(figsize=(7.5, 4.2), layout="constrained")` crea la figura de nivel de ejes con márgenes automáticos,
- **[U40.4]** `sns.ecdfplot(data=pm25_limpias, x="pm25_ug_m3", hue="zona", linewidth=2.2, ax=ax)` compara directamente las proporciones acumuladas de `Norte`, `Centro` y `Sur` en una escala común (`0` a `1`) sin artefactos de agrupamiento por intervalos,
- **[U40.4]** `ax.set(...)`, `ax.legend(title="Zona")` y `plt.show()` añaden título, unidades (`µg/m³`), leyenda legible y muestran la figura final.

### Conclusiones del Ejercicio 40.05
1. **Pauta visible:** En las lecturas simuladas limpias, las tres zonas presentan distribuciones claramente desplazadas entre sí: `Norte` concentra los valores más bajos (todos por debajo de 20 µg/m³), `Centro` ocupa una posición intermedia y `Sur` muestra la curva más desplazada hacia la derecha con las concentraciones más elevadas.
2. **Limitación de los datos:** Se trata de una muestra simulada de tamaño reducido con dos valores ausentes en `pm25_ug_m3` y sin información temporal ni meteorológica, por lo que una diferencia visual entre curvas describe únicamente estos registros y no demuestra que la ubicación geográfica cause los niveles de contaminación.
3. **Comprobación adicional necesaria:** Antes de tomar una decisión real de gestión ambiental, sería imprescindible verificar la calibración y equivalencia técnica de los sensores entre zonas, auditar el motivo de los datos ausentes y recoger series temporales más amplias bajo condiciones meteorológicas comparables.

## Autoevaluación y cierre

1. ¿Qué diferencia hay entre un valor ausente y un cero?
2. ¿Qué cambia y qué no cambia al modificar los bins de un histograma?
3. ¿Qué representa el eje vertical con `stat="density"`?
4. ¿Qué ventaja tiene una ECDF frente a un histograma?
5. ¿Por qué una diferencia entre zonas no prueba por sí sola que la zona cause los valores?

En la siguiente unidad pasarás de una variable a relaciones entre dos variables cuantitativas.